In [2]:
import pandas as pd
import sys 
import os
dataset_df = pd.read_csv('../data/processed/dataset_metadata.csv')

In [3]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root added to path: {project_root}")

Project root added to path: c:\Users\collu\Desktop\alzheimer-xai-prj


In [4]:
from src.dataset import MRIDataset
from torch.utils.data import DataLoader

# 1. Create the Dataset instance using our valid patients table
alzheimer_dataset = MRIDataset(dataset_df)

# 2. Create the DataLoader
# We use a very small batch_size (e.g., 2) because 3D images consume a lot of RAM!
# shuffle=True ensures the model doesn't memorize the patient order
train_loader = DataLoader(alzheimer_dataset, batch_size=2, shuffle=True)

# 3. Test the DataLoader by pulling the first batch
try:
    # iter() turns the loader into a stream, next() grabs the first drop
    images, labels = next(iter(train_loader))
    
    print("--- PyTorch DataLoader Test ---")
    print(f"Batch Images Shape: {images.shape}")
    print(f"Batch Labels: {labels}")

except Exception as e:
    print(f"Error loading PyTorch batch: {e}")

--- PyTorch DataLoader Test ---
Batch Images Shape: torch.Size([2, 1, 176, 208, 176])
Batch Labels: tensor([1, 1])


In [5]:
import sys
import os

# 1. Path fix to find the 'src' folder
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Import our custom module
from src.data_loader import create_mri_dataset

# 3. Define the exact paths to your raw data
excel_path = r"C:\Users\collu\Desktop\alzheimer-xai-prj\data\Demographic and Clinical Data\oasis_cross-sectional.xlsx"
base_data_dir = r"C:\Users\collu\Desktop\alzheimer-xai-prj\data"

print("--- Testing the Data Loader Pipeline ---")

try:
    # 4. Run the scanner!
    live_dataset_df = create_mri_dataset(excel_path, base_data_dir)
    
    print("\n--- Results ---")
    print(f"Total patients successfully found across all discs: {len(live_dataset_df)}")
    display(live_dataset_df.head())

except Exception as e:
    print(f"An error occurred during the scan: {e}")

--- Testing the Data Loader Pipeline ---
Loading Excel file...
Excel loaded seccessfully. Found 436 entries.
Starting 3D image search...
Processed 200 patients...
Processed 250 patients...
Processed 300 patients...
Finished searching. Found 235 valid entries with matching MRI files.

--- Results ---
Total patients successfully found across all discs: 235


,ID,Path,Label
0,OAS1_0001_MR1,C:\Users\collu\Desktop\alzheimer-xai-prj\data\...,0
1,OAS1_0002_MR1,C:\Users\collu\Desktop\alzheimer-xai-prj\data\...,0
2,OAS1_0003_MR1,C:\Users\collu\Desktop\alzheimer-xai-prj\data\...,1
3,OAS1_0010_MR1,C:\Users\collu\Desktop\alzheimer-xai-prj\data\...,0
4,OAS1_0011_MR1,C:\Users\collu\Desktop\alzheimer-xai-prj\data\...,0


In [6]:
import torch
from src.model import Alzheimer3DCNN

print("--- Testing the 3D CNN Architecture ---")

try:
    # 1. Instantiate the model
    model = Alzheimer3DCNN()
    
    # 2. Create a "fake" batch of MRIs to test the math
    # Shape: [Batch Size=2, Channels=1, Depth=176, Height=208, Width=176]
    dummy_mri_batch = torch.randn(2, 1, 176, 208, 176)
    
    # 3. Pass the fake data through the model
    output = model(dummy_mri_batch)
    
    print("Success! The model processed the 3D volume.")
    print(f"Input shape: {dummy_mri_batch.shape}")
    print(f"Output shape: {output.shape} -> (2 patients, 2 scores: [Healthy, Alzheimer])")
    print(f"Raw Output Values:\n{output}")

except Exception as e:
    print(f"Architecture Error: {e}")

--- Testing the 3D CNN Architecture ---
Success! The model processed the 3D volume.
Input shape: torch.Size([2, 1, 176, 208, 176])
Output shape: torch.Size([2, 2]) -> (2 patients, 2 scores: [Healthy, Alzheimer])
Raw Output Values:
tensor([[ 0.1654, -0.1185],
        [ 0.2882, -0.2720]], grad_fn=<AddmmBackward0>)


In [ ]:
from src.data_loader import create_mri_dataset
from src.dataset import MRIDataset
from src.model import Alzheimer3DCNN
from src.trainer import train_model
from torch.utils.data import DataLoader

# 1. Prepare Data
excel_path = r"C:\Users\collu\Desktop\alzheimer-xai-prj\data\oasis_cross-sectional.xlsx"
base_data_dir = r"C:\Users\collu\Desktop\alzheimer-xai-prj\data"

print("Loading dataset...")
dataset_df = create_mri_dataset(excel_path, base_data_dir)
alzheimer_dataset = MRIDataset(dataset_df)

# We use batch_size=2 because 3D images are huge. 
# If your computer crashes for "Out of Memory", lower this to 1!
train_loader = DataLoader(alzheimer_dataset, batch_size=2, shuffle=True)

# 2. Prepare Model
model = Alzheimer3DCNN()

# 3. Train! (We'll do just 2 epochs to test if the loss goes down)
try:
    trained_model = train_model(model, train_loader, num_epochs=2, learning_rate=0.0001)
    print("Training completely successfully!")
except Exception as e:
    print(f"An error occurred during training: {e}")